In [3]:
import torch
import numpy as np

x = torch.tensor([[1.0, 2.0], [3.0, 4.0]], dtype=torch.float32)
zeros = torch.zeros(size=(2, 3)) # массив нулей
rand = torch.randn(size=(32, 128)) # стандартное нормальное распределение n(0, 1)

# Взаимодействие с NumPy (без копирования памяти)
np_arr = np.array([1, 2, 3])
tensor_from_np = torch.from_numpy(np_arr)
back_to_np = tensor_from_np.numpy()

# изменение формы
x = torch.randn(2, 3, 4)
x_flat = x.view(2, 12) # аналог reshape
x_reshaped = x.reshape(2, 12) 

# работа с размерностями
v = torch.randn(128) # shape: [128]
v_batch = v.unsqueeze(0) # shape [1, 128] (добавили размерность батча)
print(v_batch.shape)
v_back = v_batch.squeeze(0)
print(v_back.shape)

torch.Size([1, 128])
torch.Size([128])


In [4]:
# Autograd

# 1. Флаг requires_grad=True указывает PyTorch 
# отслеживать все математические операции над тензором.

# 2. Каждый ведомый тензор хранит ссылку на операцию, которая его создала (grad_fn).

# 3. Вызов .backward() выполняет обход графа назад (Chain Rule) 
# и записывает частные производные в поле .grad исходных тензоров.

# Объявляем обучаемый параметр
w = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)

# Forward pass (строим граф вычислений)
x = torch.tensor([3.0])
y = w * x + b # = 7.0
loss = (y - 10) ** 2

# Backward pass (расчет градиентов)
loss.backward()

# Посмотрели производные: d(loss)/dw и d(loss)/db
print(w.grad) 
print(b.grad)

tensor([-18.])
tensor([-6.])


In [5]:
import torch.nn as nn

class LinearRegression(nn.Module):
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x)

model = LinearRegression(input_dim = 10, output_dim=1)

print("Список всех обучаемых параметров")
for name, param in model.named_parameters():
    print(f"Имя: {name} | Форма: {param.shape} | Requires Grad: {param.requires_grad}")

# Сохранение и загрузка состояния весов (state_dict)
weights = model.state_dict()
print('\n')
for i, j in weights.items():
    print(i, j)

Список всех обучаемых параметров
Имя: linear.weight | Форма: torch.Size([1, 10]) | Requires Grad: True
Имя: linear.bias | Форма: torch.Size([1]) | Requires Grad: True


linear.weight tensor([[ 0.2949,  0.1199, -0.1845, -0.0545, -0.2594,  0.2808, -0.0230, -0.1388,
          0.2729, -0.2464]])
linear.bias tensor([-0.2367])


In [6]:
from torch.utils.data import Dataset, DataLoader
import torch
# Pytorch Dataset & Dataloader

class CustomTextDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.labels = labels
        # Токенизируем и кодируем тексты заранее для ускорения обучения
        self.encoded_texts = [vocab.encode(text) for text in texts]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.encoded_texts[idx], self.labels[idx]

In [ ]:
def pad_collate_fn(batch):
    texts, labels = zip(*batch)

    tensor_labels = torch.tensor(labels, dtype=torch.long)
    return texts, tensor_labels

loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    collate_fn=pad_collate_fn,
    pin_memory=True
)

In [ ]:
# основной паттерн переноса данных и модели

# определение доступного устройства
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    # для мака
    device = torch.device('mps')
else:
    device = torch.device('cpu')

# перенос модели на gpu
model = MyModel().to(device)

# перенос тензоров в цикле обучения (обязательно туда же куда и модель)
for x_batch, y_batch in loader:
    x_batch = x_batch.to(device)
    y_batch = y_batch.to(device)
    output = model(x_batch)

In [ ]:
# оптимизаторы

import torch.optim as optim

# передаем оптимизатору все параметры
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

# группы параметров (Differetial Learning Rates):
# настройка разных LR или отключения Weight Decay для bias и LayerNorm
no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {
        'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
        'weight_decay': 0.01
    },
    {
        'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
        'weight_decay': 0.0
    }
]
optimizer = optim.AdamW(optimizer_grouped_parameters, lr=1e-3)

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for x_batch, y_batch in dataloader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        # сброс градиентов
        optimizer.zero_grad()

        # прямой ход
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)

        # обратный ход
        loss.backward()

        # защита от взрыва градиентов
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # обновление весов
        optimizer.step()

        total_loss / len(dataloader)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for x_batch, y_batch in dataloader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)

                outputs = model(x_batch)
                loss = criterion(outputs, y_batch)

                total_loss += loss.item()
                preds = outputs.argmax(dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(y_batch.cpu().numpy())

    return total_loss / len(dataloader), all_preds, all_targets